In [1]:
%cd /mlx_devbox/users/janne.spijkervet/repo/samantha
%load_ext autoreload
%autoreload 2

/mlx_devbox/users/janne.spijkervet/repo/samantha


In [52]:
from time import perf_counter
import torch
from recipes.soundstorm2.lightning.soundstorm import SoundStorm
from recipes.datasets.libritts import LibriTTSWebDataModule
from recipes.datasets.librilight import LibriLightWebDataModule
from samantha.utils.hdfs_helper import get, hdfs_ls
from recipes.soundstorm2.lightning.soundstream import SoundStreamSpeech24k

In [76]:
sample_rate = 16000
pl_datamodule = LibriTTSWebDataModule(
    sample_rate=sample_rate,
    batch_size=8,
    shuffle_buffer_size=100,
)

# pl_datamodule = LibriLightWebDataModule(
#     sample_rate=16000,
#     split="large",
#     batch_size=8,
#     shuffle_buffer_size=100,
# )



train_loader = pl_datamodule.train_dataloader()
batch = next(iter(train_loader))

cat: Unable to write to output stream.
cat: Unable to write to output stream.
cat: Unable to write to output stream.
cat: Unable to write to output stream.
cat: Unable to write to output stream.
cat: Unable to write to output stream.
cat: Unable to write to output stream.
cat: Unable to write to output stream.


In [59]:
device = "cuda"

audio_model = SoundStreamSpeech24k().to(device)
soundstorm = SoundStorm.load_from_checkpoint("epoch=0-step=154000.ckpt", audio_model=audio_model).to(device)

cat: Unable to write to output stream.


In [60]:
with torch.no_grad():
    audio_tokens = soundstorm.audio_model(batch["audio"].to(device))

In [61]:
print(audio_tokens.shape)

torch.Size([8, 12, 318])


In [62]:
iterations = [48, 32, 24, 16, 8, 4, 2, 2, 1, 1, 1, 1]
score_strategies = [
    "random",
    "random",
    "random",
    "random",
    "maskgit",
    "maskgit",
    "maskgit",
    "maskgit",
    "maskgit",
    "maskgit",
    "maskgit",
    "maskgit",
]
guidance_scale = None
temperatures = [1.0, 1.0, 0.95, 0.95, 0.9, 0.9, 0.8, 0.8, 0.4, 0.4, 0.4, 0.4]
sampled_t = torch.randint(50, 100, (1,), device=soundstorm.device)

In [73]:
sampled_audio_tokens, _ = soundstorm.iterative_decoding(
    max_seq_len=audio_tokens.shape[2],
    iterations=iterations,
    score_strategies=score_strategies,
    guidance_scale=guidance_scale,
    temperatures=temperatures,
    sampled_t=sampled_t,
    seed_tokens=audio_tokens,
    prefix_tokens=None,
    semantic_tokens=None,
)

Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:20<00:00,  1.73s/it]


In [74]:
with torch.no_grad():
    dec_audio = soundstorm.audio_model.decode(audio_tokens)
    sampled_audio = soundstorm.audio_model.decode(sampled_audio_tokens)


In [75]:
from IPython import display as ipd

ipd.display(ipd.Audio(batch["audio"][0].cpu(), rate=16000))
ipd.display(ipd.Audio(dec_audio[0].cpu(), rate=16000))
ipd.display(ipd.Audio(sampled_audio[0].cpu(), rate=16000))

for idx in range(4):
    ipd.display(ipd.Audio(sampled_audio[idx].cpu(), rate=16000))